In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import time


In [3]:
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
cards_xpath = "//a/card"

def scroll_smooth(height_step=500):
    """Scroll suave em incrementos para carregar mais conteúdo"""
    current_height = driver.execute_script("return window.pageYOffset;")
    new_height = current_height + height_step
    driver.execute_script(f"window.scrollTo(0, {new_height});")

def collect_cards(max_items=1000, timeout=300):
    start = time.time()
    cards = driver.find_elements(By.XPATH, cards_xpath)
    last_count = len(cards)
    scroll_attempts = 0

    while len(cards) < max_items and (time.time() - start) < timeout and scroll_attempts < 50:
        # Scroll suave em incrementos
        for _ in range(10):  # 10 scrolls de 500px cada
            scroll_smooth(500)
            time.sleep(0.5)

        # Scroll até bottom para garantir carregamento
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        cards = driver.find_elements(By.XPATH, cards_xpath)
        scroll_attempts += 1

        if len(cards) == last_count:
            print(f"Nenhum novo card após {scroll_attempts} scrolls. Parando.")
            break

        last_count = len(cards)
        print(f"Scroll {scroll_attempts}: {len(cards)} cards")

        if len(cards) >= max_items:
            break

    return cards[:max_items]

In [4]:

driver.get("https://www.seminovosmovida.com.br/busca")


In [5]:

# XPath de todos os cards listados

cards = collect_cards(max_items=1000, timeout=120)
print("Total de cards coletados:", len(cards))


Scroll 1: 60 cards
Scroll 2: 100 cards
Scroll 3: 140 cards
Scroll 4: 180 cards
Scroll 5: 220 cards
Scroll 6: 260 cards
Scroll 7: 300 cards
Scroll 8: 340 cards
Scroll 9: 380 cards
Scroll 10: 420 cards
Scroll 11: 460 cards
Scroll 12: 500 cards
Scroll 13: 540 cards
Scroll 14: 580 cards
Scroll 15: 620 cards
Scroll 16: 660 cards
Scroll 17: 700 cards
Total de cards coletados: 700


In [9]:

dados = [c.text.split("\n") for c in cards]
# exemplo: exibir primeiro resultado
if dados:
    print("Primeiro card:", dados[0])

# quando terminar
driver.quit()

MaxRetryError: HTTPConnectionPool(host='localhost', port=55908): Max retries exceeded with url: /session/d1c711d10b482d695928b860aec48cfb/element/f.BD0FAB1D35F1E47C02A7BBEECAA9A828.d.D9649FB7FED17F9FE73F64586867BA11.e.252/text (Caused by NewConnectionError("HTTPConnection(host='localhost', port=55908): Failed to establish a new connection: [WinError 10061] Nenhuma conexão pôde ser feita porque a máquina de destino as recusou ativamente"))

In [7]:
# Inspecionar estrutura dos dados coletados
print(f"Total de cards: {len(cards)}")
print(f"Total de dados: {len(dados) if 'dados' in globals() else 'dados não definido'}")

if 'dados' in globals() and dados:
    print("\nEstrutura do primeiro card:")
    for i, linha in enumerate(dados[0][:10]):  # primeiras 10 linhas
        print(f"{i}: '{linha}'")

    print(f"\nTotal de linhas no primeiro card: {len(dados[0])}")

    # Verificar se todos têm mesma estrutura
    estruturas = [len(d) for d in dados[:5]]
    print(f"Estruturas dos primeiros 5 cards: {estruturas}")

else:
    print("Dados não disponíveis. Execute a coleta primeiro.")

# Quando dados estiverem prontos, salvar para Power BI
import pandas as pd
import re

def salvar_para_powerbi(dados_cards, filename="carros_movida.csv"):
    """
    Salva dados dos cards em formato ideal para Power BI com colunas estruturadas
    """
    if not dados_cards:
        print("Nenhum dado para salvar")
        return

    registros = []

    for i, card in enumerate(dados_cards):
        if len(card) >= 4:
            # Extrair informações da linha 2 (Ano/Modelo/KM/Local)
            linha_info = card[2] if len(card) > 2 else ''

            # Usar regex para extrair dados
            ano_match = re.search(r'Ano/Modelo (\d{4})/(\d{4})', linha_info)
            km_match = re.search(r'(\d+(?:\.\d+)?)\s*KM', linha_info)
            local_match = re.search(r'•\s*(.+)$', linha_info)

            registro = {
                'id': i + 1,
                'marca': card[0].split()[0] if card[0] else '',  # Primeira palavra
                'modelo': card[0] if len(card) > 0 else '',
                'versao': card[1] if len(card) > 1 else '',
                'ano_fabricacao': ano_match.group(1) if ano_match else '',
                'ano_modelo': ano_match.group(2) if ano_match else '',
                'km': km_match.group(1).replace('.', '') if km_match else '',
                'localizacao': local_match.group(1) if local_match else '',
                'preco_bruto': card[3] if len(card) > 3 else '',
                'condicoes': card[4] if len(card) > 4 else '',
                'texto_completo': '\n'.join(card),
                'data_coleta': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
            }
            registros.append(registro)

    df = pd.DataFrame(registros)

    # Limpar e converter dados
    df['preco'] = df['preco_bruto'].str.replace('R\$', '').str.replace('.', '').str.replace(',', '.').str.extract('(\d+\.?\d*)').astype(float)
    df['km'] = pd.to_numeric(df['km'], errors='coerce')
    df['ano_fabricacao'] = pd.to_numeric(df['ano_fabricacao'], errors='coerce')
    df['ano_modelo'] = pd.to_numeric(df['ano_modelo'], errors='coerce')

    # Adicionar colunas calculadas para Power BI
    df['faixa_preco'] = pd.cut(df['preco'],
                              bins=[0, 30000, 50000, 80000, 120000, float('inf')],
                              labels=['Até 30k', '30k-50k', '50k-80k', '80k-120k', '120k+'])

    df['faixa_km'] = pd.cut(df['km'],
                           bins=[0, 30000, 60000, 100000, 150000, float('inf')],
                           labels=['Até 30k km', '30k-60k km', '60k-100k km', '100k-150k km', '150k+ km'])

    df['idade_veiculo'] = 2026 - df['ano_fabricacao']  # ano atual aproximado

    # Salvar
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"✅ Dados salvos em {filename}")
    print(f"📊 Colunas criadas: {list(df.columns)}")
    print(f"📈 Total registros: {len(df)}")
    print(f"💰 Preço médio: R$ {df['preco'].mean():.2f}")
    print(f"🛣️ KM médio: {df['km'].mean():.0f}")

    return df

# Salvar dados atuais
if 'dados' in globals() and dados:
    df_carros = salvar_para_powerbi(dados, "carros_movida_260.csv")
else:
    print("Execute a coleta completa primeiro para salvar todos os 1000 carros.")

<>:62: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<>:62: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:62: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<>:62: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\nahuan\AppData\Local\Temp\ipykernel_12796\2021240515.py:62: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
  df['preco'] = df['preco_bruto'].str.replace('R\$', '').str.replace('.', '').str.replace(',', '.').str.extract('(\d+\.?\d*)').astype(float)
C:\Users\nahuan\AppData\Loc

Total de cards: 700
Total de dados: 700

Estrutura do primeiro card:
0: 'Fiat Uno'
1: 'Attractive 1.0 Fire Flex 8v 5p'
2: 'Ano/Modelo 2020/2021 • 82.817 KM • São Paulo Vila Carrão, SP'
3: 'R$ 46.200'
4: 'Entrada + parcelas a partir de R$ 684*'

Total de linhas no primeiro card: 5
Estruturas dos primeiros 5 cards: [5, 5, 5, 5, 5]
✅ Dados salvos em carros_movida_260.csv
📊 Colunas criadas: ['id', 'marca', 'modelo', 'versao', 'ano_fabricacao', 'ano_modelo', 'km', 'localizacao', 'preco_bruto', 'condicoes', 'texto_completo', 'data_coleta', 'preco', 'faixa_preco', 'faixa_km', 'idade_veiculo']
📈 Total registros: 700
💰 Preço médio: R$ 55514.33
🛣️ KM médio: 65456


In [8]:
# 📊 IDEIAS CRIATIVAS PARA POWER BI

print("""
🎨 DASHBOARD POWER BI - CARROS MOVIDA

📁 Arquivo gerado: carros_movida_260.csv

📊 VISUALIZAÇÕES SUGERIDAS:

1️⃣ 📈 DISTRIBUIÇÃO DE PREÇOS
   - Histograma de preços com curva de densidade
   - Box plot por marca
   - Mapa de calor: Preço vs KM vs Ano

2️⃣ 🗺️ DISTRIBUIÇÃO GEOGRÁFICA
   - Mapa com pins por localização
   - Treemap por estado/cidade
   - Barras empilhadas: Marca vs Localização

3️⃣ 📊 ANÁLISE TEMPORAL
   - Linha do tempo: Preço médio por ano de fabricação
   - Scatter plot: Ano vs Preço (com tamanho = KM)
   - Idade média dos veículos por marca

4️⃣ 🚗 SEGMENTAÇÃO AVANÇADA
   - Sunburst: Marca > Modelo > Versão
   - Matriz de correlação: Preço, KM, Ano
   - KPI Cards: Total veículos, Preço médio, KM médio

5️⃣ 💡 INSIGHTS CRIATIVOS
   - Preço por KM rodado (eficiência)
   - Marcas mais econômicas vs premium
   - Localizações com melhores ofertas
   - Tendências de quilometragem por ano

🎯 DICAS POWER BI:
- Use segmentação por: Marca, Faixa de Preço, Faixa KM, Localização
- Crie medidas DAX: Preço médio por marca, % veículos novos, etc.
- Adicione tooltips ricos com fotos e detalhes
- Use temas escuros para visual moderno
- Publique no Power BI Service para compartilhamento

📈 MÉTRICAS CALCULADAS POSSÍVEIS:
- Preço/KM = Preço ÷ KM
- Desvalorização = (Preço estimado - Preço atual) ÷ Preço estimado
- Score de oportunidade = baseado em preço vs mercado
""")

# Verificar se arquivo foi criado
import os
if os.path.exists("carros_movida_260.csv"):
    print("✅ Arquivo CSV criado com sucesso!")
    print(f"📂 Localização: {os.path.abspath('carros_movida_260.csv')}")
else:
    print("❌ Arquivo não encontrado")


🎨 DASHBOARD POWER BI - CARROS MOVIDA

📁 Arquivo gerado: carros_movida_260.csv

📊 VISUALIZAÇÕES SUGERIDAS:

1️⃣ 📈 DISTRIBUIÇÃO DE PREÇOS
   - Histograma de preços com curva de densidade
   - Box plot por marca
   - Mapa de calor: Preço vs KM vs Ano

2️⃣ 🗺️ DISTRIBUIÇÃO GEOGRÁFICA
   - Mapa com pins por localização
   - Treemap por estado/cidade
   - Barras empilhadas: Marca vs Localização

3️⃣ 📊 ANÁLISE TEMPORAL
   - Linha do tempo: Preço médio por ano de fabricação
   - Scatter plot: Ano vs Preço (com tamanho = KM)
   - Idade média dos veículos por marca

4️⃣ 🚗 SEGMENTAÇÃO AVANÇADA
   - Sunburst: Marca > Modelo > Versão
   - Matriz de correlação: Preço, KM, Ano
   - KPI Cards: Total veículos, Preço médio, KM médio

5️⃣ 💡 INSIGHTS CRIATIVOS
   - Preço por KM rodado (eficiência)
   - Marcas mais econômicas vs premium
   - Localizações com melhores ofertas
   - Tendências de quilometragem por ano

🎯 DICAS POWER BI:
- Use segmentação por: Marca, Faixa de Preço, Faixa KM, Localização
- Cri

In [15]:
# Reiniciar driver e debug
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://www.seminovosmovida.com.br/busca")
time.sleep(5)  # esperar página carregar
initial_cards = driver.find_elements(By.XPATH, cards_xpath)
print(f"Cards iniciais: {len(initial_cards)}")

# Testar scroll manual
scroll_to_bottom()
time.sleep(3)
after_scroll_cards = driver.find_elements(By.XPATH, cards_xpath)
print(f"Cards após primeiro scroll: {len(after_scroll_cards)}")

# Verificar se há botão "carregar mais" ou similar
try:
    load_more = driver.find_element(By.XPATH, "//button[contains(text(), 'Carregar mais')]")
    print("Botão 'Carregar mais' encontrado!")
except:
    print("Nenhum botão 'Carregar mais' encontrado.")

# Verificar estrutura dos cards
if initial_cards:
    print("Estrutura do primeiro card:")
    print(initial_cards[0].get_attribute('outerHTML')[:500])  # primeiros 500 chars

# Testar XPath alternativo
alt_xpath = "//div[contains(@class, 'card')]"  # ou ajustar conforme necessário
alt_cards = driver.find_elements(By.XPATH, alt_xpath)
print(f"Cards com XPath alternativo '{alt_xpath}': {len(alt_cards)}")

driver.quit()

Cards iniciais: 20
Cards após primeiro scroll: 40
Nenhum botão 'Carregar mais' encontrado.
Estrutura do primeiro card:
<card _ngcontent-ng-c3939050943="" _nghost-ng-c4200686794=""><div _ngcontent-ng-c4200686794="" class="card text-center"><div _ngcontent-ng-c4200686794="" class="card__inner"><div _ngcontent-ng-c4200686794="" class="card__image"><img _ngcontent-ng-c4200686794="" class="vehicle-img" src="assets/img/em_preparacao.webp" alt="imagem carro Fiat Uno"><div _ngcontent-ng-c4200686794="" class="selos-container-promotion"><!----><div _ngcontent-ng-c4200686794="" class="selos-bottom"><img _ngcontent-ng-c4200
Cards com XPath alternativo '//div[contains(@class, 'card')]': 160


In [16]:
# Teste: scroll múltiplo para ver limite
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://www.seminovosmovida.com.br/busca")
time.sleep(5)

cards_xpath = "//a/card"  # manter original

def scroll_smooth(height_step=500):
    """Scroll suave em incrementos"""
    current_height = driver.execute_script("return window.pageYOffset;")
    new_height = current_height + height_step
    driver.execute_script(f"window.scrollTo(0, {new_height});")

def collect_with_smooth_scroll(max_items=1000, timeout=300):
    start = time.time()
    cards = driver.find_elements(By.XPATH, cards_xpath)
    last_count = len(cards)
    scroll_attempts = 0

    while len(cards) < max_items and (time.time() - start) < timeout and scroll_attempts < 50:
        # Scroll suave
        for _ in range(10):  # scroll 10x de 500px
            scroll_smooth(500)
            time.sleep(0.5)

        # Scroll até bottom para garantir
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        cards = driver.find_elements(By.XPATH, cards_xpath)
        scroll_attempts += 1

        if len(cards) == last_count:
            print(f"Nenhum novo card após {scroll_attempts} scrolls. Parando.")
            break

        last_count = len(cards)
        print(f"Scroll {scroll_attempts}: {len(cards)} cards")

        if len(cards) >= max_items:
            break

    return cards[:max_items]

cards = collect_with_smooth_scroll(max_items=1000, timeout=300)
print(f"Total coletado: {len(cards)}")

driver.quit()

Scroll 1: 60 cards
Scroll 2: 100 cards
Scroll 3: 140 cards
Scroll 4: 180 cards
Scroll 5: 220 cards
Scroll 6: 260 cards
Scroll 7: 300 cards
Scroll 8: 340 cards
Scroll 9: 380 cards
Scroll 10: 400 cards
Scroll 11: 460 cards
Scroll 12: 500 cards
Scroll 13: 540 cards
Scroll 14: 580 cards
Scroll 15: 620 cards
Scroll 16: 660 cards
Scroll 17: 700 cards
Scroll 18: 740 cards
Scroll 19: 780 cards
Scroll 20: 820 cards
Scroll 21: 860 cards
Scroll 22: 900 cards
Scroll 23: 940 cards
Scroll 24: 980 cards
Scroll 25: 1020 cards
Total coletado: 1000
